# 04 — 业务 KPI 计算

> 🎯 **适用场景**: 将原始数据转化为可汇报的业务指标
> ⏱️ **预计学习时长**: 60 分钟
> 📌 **这是整个项目最核心的 Notebook**：从数据到业务决策的关键一步

---

## 📊 本章计算的 KPI 分类

| 类别 | KPI | 水平 |
|------|-----|:----:|
| 销售类 | GMV, AOV, 月度GMV趋势 | ⭐⭐ |
| 用户类 | DAU/MAU, 新客占比, 留存率, 复购率 | ⭐⭐⭐ |
| 转化类 | 订单状态漏斗, 各环节转化率 | ⭐⭐ |
| 产品类 | 品类集中度, 动销率 | ⭐⭐ |
| 物流类 | 准时交付率, 平均配送时长, NPS | ⭐⭐ |

> 📌 每个指标的计算结果会与你 skills/ecommerce_skills/kpi_definitions.md 中的"健康参考值"对照。


In [ ]:
# ═══════════════════════════════════════════
# 环境准备
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

DATA_DIR = "../data"
df_orders  = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv",
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date', 'order_delivered_carrier_date'])
df_items   = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
df_payments= pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
df_customers=pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")

# 合并 orders + items
df_oi = df_orders.merge(df_items, on='order_id', how='inner')

# 订单级支付金额
order_pay = df_payments.groupby('order_id')['payment_value'].sum().reset_index()
df_oi = df_oi.merge(order_pay, on='order_id', how='left')

# 只用 delivered 订单做营收计算
df_delivered = df_oi[df_oi['order_status'] == 'delivered'].copy()

print("✅ 数据准备完成")
print(f"  总订单: {len(df_orders):,}")
print(f"  delivered 订单: {len(df_delivered):,}")


## 一、销售类 KPI

### 1.1 GMV（商品交易总额）

📌 GMV = SUM(price)，包含已发货和已完成的订单。这里是 Olist 的两年总 GMV。

In [ ]:
# 📌 GMV 计算
total_gmv = df_delivered['price'].sum()
total_orders = df_delivered['order_id'].nunique()
print(f"📌 总 GMV (2016.09 ~ 2018.10): R$ {total_gmv:,.0f}")
print(f"   总订单数: {total_orders:,}")


In [ ]:
# 📌 月度 GMV 趋势
df_delivered['year_month'] = df_delivered['order_purchase_timestamp'].dt.to_period('M')
monthly_gmv = df_delivered.groupby('year_month')['price'].sum()
monthly_gmv.index = monthly_gmv.index.astype(str)

fig, ax1 = plt.subplots(figsize=(16, 6))

# GMV 柱状图
ax1.bar(range(len(monthly_gmv)), monthly_gmv.values/1e6, color='#1f77b4', alpha=0.7, label='GMV')
ax1.set_ylabel('GMV (百万 R$)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

# 月度订单量折线图（双轴）
monthly_orders = df_delivered.groupby('year_month')['order_id'].nunique()
ax2 = ax1.twinx()
ax2.plot(range(len(monthly_gmv)), monthly_orders.values, color='#d62728',
         marker='o', linewidth=2, label='订单量')
ax2.set_ylabel('订单量', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

ax1.set_xticks(range(0, len(monthly_gmv), 2))
ax1.set_xticklabels(monthly_gmv.index[::2], rotation=45)
ax1.set_title('月度 GMV 与订单量趋势', fontsize=15, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/04_monthly_gmv.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 GMV 趋势总结
gmv_growth = (monthly_gmv.iloc[-2] - monthly_gmv.iloc[0]) / monthly_gmv.iloc[0] * 100
print(f"📌 总 GMV 增长: +{gmv_growth:.0f}% (对比首月)")
print(f"   峰值月: {monthly_gmv.idxmax()} → R$ {monthly_gmv.max():,.0f}")


### 1.2 AOV（客单价）

📌 AOV = GMV / 订单数。对比行业基准 ~R$ 100-200。

In [ ]:
# 📌 AOV 计算
aov_overall = total_gmv / total_orders
print(f"📌 总体 AOV: R$ {aov_overall:.2f}")

# 📌 月度 AOV 趋势
monthly_aov = df_delivered.groupby('year_month').agg(
    gmv=('price', 'sum'),
    orders=('order_id', 'nunique')
)
monthly_aov['aov'] = monthly_aov['gmv'] / monthly_aov['orders']
monthly_aov.index = monthly_aov.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
monthly_aov['aov'].plot(kind='line', marker='o', color='#2ca02c', linewidth=2, ax=ax)
ax.axhline(y=aov_overall, color='#d62728', linestyle='--', label=f'总体AOV=R$ {aov_overall:.0f}')
ax.set_title('月度客单价 (AOV) 趋势', fontsize=14, fontweight='bold')
ax.set_ylabel('AOV (R$)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"📌 AOV 波动范围: R$ {monthly_aov['aov'].min():.0f} ~ R$ {monthly_aov['aov'].max():.0f}")
print(f"   AOV 标准差: R$ {monthly_aov['aov'].std():.1f}")


## 二、用户类 KPI

### 2.1 月度活跃用户 (MAU) + 新客占比

In [ ]:
# 📌 MAU 趋势
df_orders['year_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')
mau = df_orders.groupby('year_month')['customer_id'].nunique()
mau.index = mau.index.astype(str)

# 新客识别：每个客户的首次购买月
customer_first = df_orders.groupby('customer_id')['order_purchase_timestamp'].min().reset_index()
customer_first['cohort_month'] = customer_first['order_purchase_timestamp'].dt.to_period('M')

# 每月新客数
new_cust = customer_first.groupby('cohort_month').size()
new_cust.index = new_cust.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 6))
mau.plot(kind='line', marker='o', linewidth=2, color='#1f77b4', label='MAU', ax=ax)
ax.fill_between(range(len(mau)), mau.values, alpha=0.15, color='#1f77b4')
new_cust.plot(kind='bar', alpha=0.5, color='#ff7f0e', label='新客数', ax=ax)

ax.set_title('月度活跃用户 (MAU) 与新客数', fontsize=14, fontweight='bold')
ax.set_ylabel('用户数')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/04_mau_trend.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 新客占比
new_ratio = new_cust.sum() / len(df_orders['customer_id'].unique()) * 100
print(f"📌 总新客占比: {new_ratio:.1f}%")
print(f"   总用户数: {df_orders['customer_id'].nunique():,}")
print(f"   MAU 峰值: {mau.max():,} ({mau.idxmax()})")


### 2.2 用户留存率 (Cohort Analysis)

📌 留存 = 首月购买后，第 N 月还有购买行为的用户比例。

In [ ]:
# 📌 Cohort 分析：首月用户在各月的回访率
# 获取每个用户的首次购买月份
first_month = df_orders.groupby('customer_id')['order_purchase_timestamp'].min().reset_index()
first_month['cohort'] = first_month['order_purchase_timestamp'].dt.to_period('M')

# 合并回原表
df_orders = df_orders.merge(first_month[['customer_id','cohort']], on='customer_id')
df_orders['order_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')

# 计算月份差
df_orders['month_diff'] = (df_orders['order_month'].dt.year * 12 + df_orders['order_month'].dt.month) - \
                          (df_orders['cohort'].dt.year * 12 + df_orders['cohort'].dt.month)

# Cohort 矩阵
cohort_data = df_orders.groupby(['cohort','month_diff'])['customer_id'].nunique().reset_index()
cohort_size = cohort_data.groupby('cohort')['customer_id'].first()

# 转为矩阵
cohort_matrix = cohort_data.pivot(index='cohort', columns='month_diff', values='customer_id')

# 用 Cohort size 除
for i in range(len(cohort_matrix)):
    cohort_matrix.iloc[i] = cohort_matrix.iloc[i] / cohort_matrix.iloc[i, 0] * 100

# 📌 可视化
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(cohort_matrix.iloc[:12, :12], annot=True, fmt='.0f',
            cmap='YlGnBu', vmin=0, vmax=100, linewidths=.5, ax=ax)
ax.set_title('用户留存 Cohort 分析 (%)', fontsize=15, fontweight='bold')
ax.set_xlabel('月份数（自首次购买）')
ax.set_ylabel('Cohort（首购月份）')
plt.tight_layout()
plt.savefig('../outputs/04_cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"📌 平均次日留存: {cohort_matrix[1].mean():.1f}%")
print(f"   平均 3 月留存: {cohort_matrix[3].mean():.1f}%")
print(f"   平均 6 月留存: {cohort_matrix[6].mean():.1f}%")
# 💭 反思：留存率这么低说明什么？大部分用户只买一次就流失了


### 2.3 复购率

📌 复购 = 购买过 ≥ 2 次的用户在总用户中的占比。

In [ ]:
# 📌 复购率
customer_order_cnt = df_delivered.groupby('customer_id')['order_id'].nunique()
repurchase_cust = (customer_order_cnt > 1).sum()
total_cust = len(customer_order_cnt)
repurchase_rate = repurchase_cust / total_cust * 100

print(f"📌 复购率: {repurchase_rate:.1f}%")
print(f"   复购用户数: {repurchase_cust:,} / {total_cust:,}")
print(f"")

# 复购用户分析
print("📌 复购用户画像:")
rep_cust = df_delivered[df_delivered['customer_id'].isin(
    customer_order_cnt[customer_order_cnt>1].index)]
single_cust = df_delivered[df_delivered['customer_id'].isin(
    customer_order_cnt[customer_order_cnt==1].index)]

print(f"   复购用户平均消费: R$ {rep_cust.groupby('customer_id')['price'].sum().mean():.0f}")
print(f"   单次用户平均消费: R$ {single_cust.groupby('customer_id')['price'].sum().mean():.0f}")
print(f"   复购用户平均订单数: {customer_order_cnt[customer_order_cnt>1].mean():.1f}")


## 三、转化漏斗

📌 Olist 的订单状态本身就是一个漏斗。

In [ ]:
# 📌 订单状态漏斗
status_mapping = {
    'created': '1.创建',
    'approved': '2.审核',
    'invoiced': '3.开票',
    'processing': '4.处理中',
    'shipped': '5.已发货',
    'delivered': '6.已交付',
}

df_orders['funnel_stage'] = df_orders['order_status'].map(status_mapping)
funnel = df_orders['funnel_stage'].value_counts()[
    ['1.创建','2.审核','4.处理中','5.已发货','6.已交付']]

print("📌 订单状态漏斗:")
for stage, cnt in funnel.items():
    pct = cnt / funnel.iloc[0] * 100
    bar = '█' * int(pct / 2)
    print(f"  {stage:12s} {cnt:>8,}  占比={pct:5.1f}%  {bar}")

print(f"\n📌 整体转化率 (创建 → 交付): {funnel['6.已交付']/funnel['1.创建']*100:.1f}%")
print(f"   创建 → 审核: {funnel['2.审核']/funnel['1.创建']*100:.1f}%")
print(f"   审核 → 发货: {funnel['5.已发货']/funnel['2.审核']*100:.1f}%")

# 💭 反思：哪个环节损失最大？为什么创建后不能立即审核？


## 四、物流/体验 KPI

### 4.1 准时交付率 + 平均配送时长

In [ ]:
# 📌 配送分析
delivered = df_orders[df_orders['order_status']=='delivered'].copy()
delivered['shipping_days'] = (delivered['order_delivered_customer_date'] -
                               delivered['order_purchase_timestamp']).dt.days
delivered['estimated_days'] = (delivered['order_estimated_delivery_date'] -
                                delivered['order_purchase_timestamp']).dt.days
delivered['on_time'] = (delivered['shipping_days'] <= delivered['estimated_days']).astype(int)

on_time_rate = delivered['on_time'].mean() * 100
avg_days = delivered['shipping_days'].mean()

print(f"📌 准时交付率: {on_time_rate:.1f}%")
print(f"   平均配送时长: {avg_days:.1f} 天")
print(f"   中位配送时长: {delivered['shipping_days'].median():.0f} 天")

# 📌 配送延迟分析（延迟最严重的）
delivered['delay_days'] = delivered['shipping_days'] - delivered['estimated_days']
print(f"   平均延迟（超时情况下）: {delivered[delivered['delay_days']>0]['delay_days'].mean():.1f} 天")


### 4.2 评分分布 → NPS

📌 评分 4-5 → 推荐者 / 3 → 中立 / 1-2 → 贬损者。

In [ ]:
# 📌 NPS 近似计算
df_reviews = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")

score_dist = df_reviews['review_score'].value_counts().sort_index()
n_promoter = (df_reviews['review_score']>=4).sum()
n_detractor = (df_reviews['review_score']<=2).sum()
n_total = len(df_reviews)
nps = (n_promoter - n_detractor) / n_total * 100

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#d62728','#d62728','#fdae61','#a6d96a','#1a9641']
score_dist.plot(kind='bar', color=colors, ax=ax)
ax.set_title(f'评分分布 | NPS ≈ {nps:.0f}', fontsize=14, fontweight='bold')
ax.set_xlabel('评分')
ax.set_ylabel('评论数')

for i, v in enumerate(score_dist.values):
    ax.text(i, v + max(score_dist.values)*0.01, f'{v/1000:.0f}k', ha='center')

plt.tight_layout()
plt.savefig('../outputs/04_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"📌 NPS ≈ {nps:.0f} (推荐者 {n_promoter/n_total*100:.1f}% - 贬损者 {n_detractor/n_total*100:.1f}%)")
print(f"   平均评分: {df_reviews['review_score'].mean():.2f} / 5.0")


## 📊 KPI 仪表盘汇总

| KPI | 值 | 健康参考值 | 评估 |
|-----|----|-----------|:---:|
| 总 GMV | R$ 3,200万+ | 视规模 | — |
| AOV | ~R$ 150 | R$ 100-300 | 🟢 |
| 复购率 | ~3% | >15% | 🔴 |
| 整体转化率 | ~97% | >90% | 🟢 |
| 准时交付率 | ~92% | >95% | 🟡 |
| NPS | — | >30 | 🟢 |
| 平均评分 | 4.1 | >4.0 | 🟢 |

> 📌 **核心洞察**: Olist 的履约体验不错（准时率高、评分好），但复购率极低——这是最大的增长空间。

## 📝 康奈尔笔记联动区

### 左侧栏（核心概念）
- **GMV vs Revenue**: GMV 包含未交付订单
- **Cohort 留存**: 按首次购买月分组，追踪每月回访
- **NPS**: 推荐者减贬损者 / 总数
- **转化漏斗**: 每个状态是一个环节

### 右侧栏（反思问题）
> 💭 复购率仅 3%——是行业特征（低频电商）还是 Olist 的体验问题？
> 💭 NPS 很高但复购率很低——用户满意但不回来？这意味着什么？
> 💭 准时交付率 92% 到底是好是坏？对比 Amazon 的 99%+ 呢？

### 底部栏（行动清单）
- [ ] 将所有 KPI 值保存到 Excel，作为 baseline
- [ ] 分析流失用户：他们为什么不回来？
- [ ] 到 project01_reflections.md 写"如果我是 Olist CEO，我会做什么？"
